In [1]:
import os

print(os.listdir("/kaggle/input"))

['notebooks']


In [2]:
import numpy as np
import pandas as pd

import os
import json
import joblib
import warnings


warnings.filterwarnings("ignore")


from sklearn.model_selection import train_test_split


from sklearn.preprocessing import StandardScaler


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


from sklearn.ensemble import (
    RandomForestClassifier,
    IsolationForest
)


from xgboost import XGBClassifier


for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(
            os.path.join(dirname, filename)
        )

/kaggle/input/notebooks/harder123/notebook1/label_encoder.pkl
/kaggle/input/notebooks/harder123/notebook1/__results__.html
/kaggle/input/notebooks/harder123/notebook1/__notebook__.ipynb
/kaggle/input/notebooks/harder123/notebook1/__output__.json
/kaggle/input/notebooks/harder123/notebook1/ids2018_clean.parquet
/kaggle/input/notebooks/harder123/notebook1/custom.css


In [3]:
import pandas as pd
import joblib


DATA_PATH = "/kaggle/input/notebooks/harder123/notebook1/ids2018_clean.parquet"

ENCODER_PATH = "/kaggle/input/notebooks/harder123/notebook1/label_encoder.pkl"


df = pd.read_parquet(DATA_PATH)


encoder = joblib.load(
    ENCODER_PATH
)

In [4]:
df.shape

(823346, 79)

In [5]:
for i,name in enumerate(
    encoder.classes_
):
    print(i,name)

0 Benign
1 Bot
2 Brute Force -Web
3 Brute Force -XSS
4 DDOS attack-HOIC
5 DDOS attack-LOIC-UDP
6 DDoS attacks-LOIC-HTTP
7 DoS attacks-GoldenEye
8 DoS attacks-Hulk
9 DoS attacks-SlowHTTPTest
10 DoS attacks-Slowloris
11 FTP-BruteForce
12 Infilteration
13 SQL Injection
14 SSH-Bruteforce


In [6]:
X = df.drop(
    "Label",
    axis=1
)


y = df["Label"]

In [7]:
X.shape

(823346, 78)

In [8]:
features=list(
    X.columns
)


joblib.dump(
    features,
    "/kaggle/working/features.pkl"
)

['/kaggle/working/features.pkl']

In [9]:
X_train,X_test,y_train,y_test=train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y

)

In [10]:
print(X_train.shape)

print(X_test.shape)

(658676, 78)
(164670, 78)


In [11]:
scaler = StandardScaler()


X_train_scaled=scaler.fit_transform(
    X_train
)


X_test_scaled=scaler.transform(
    X_test
)

In [12]:
joblib.dump(
    scaler,
    "/kaggle/working/scaler.pkl"
)

['/kaggle/working/scaler.pkl']

In [13]:
rf = RandomForestClassifier(

    n_estimators=200,

    max_depth=25,

    n_jobs=-1,

    random_state=42

)

In [14]:
rf.fit(
    X_train,
    y_train
)

RandomForestClassifier(max_depth=25, n_estimators=200, n_jobs=-1,
                       random_state=42)

In [15]:
rf_pred=rf.predict(
    X_test
)

In [16]:
print(
classification_report(
    y_test,
    rf_pred
)
)

              precision    recall  f1-score   support

           0       0.96      0.99      0.97    137505
           1       1.00      1.00      1.00      2900
           2       1.00      0.45      0.62        11
           3       0.67      0.67      0.67         3
           4       1.00      1.00      1.00      6939
           5       1.00      1.00      1.00        34
           6       1.00      0.99      1.00      1428
           7       1.00      1.00      1.00       793
           8       1.00      1.00      1.00      5736
           9       0.36      0.44      0.40         9
          10       1.00      1.00      1.00       203
          11       0.29      0.25      0.27         8
          12       0.50      0.15      0.23      7287
          13       1.00      0.50      0.67         2
          14       1.00      1.00      1.00      1812

    accuracy                           0.96    164670
   macro avg       0.85      0.76      0.79    164670
weighted avg       0.94   

In [17]:
joblib.dump(
    rf,
    "/kaggle/working/random_forest.pkl"
)

['/kaggle/working/random_forest.pkl']

In [18]:
xgb = XGBClassifier(

    n_estimators=300,

    max_depth=10,

    learning_rate=0.1,

    tree_method="hist",

    eval_metric="mlogloss",

    random_state=42

)

In [19]:
xgb.fit(
    X_train,
    y_train
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=10, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [20]:
xgb_pred=xgb.predict(
    X_test
)

In [21]:
print(
classification_report(
    y_test,
    xgb_pred
)
)

              precision    recall  f1-score   support

           0       0.96      0.99      0.98    137505
           1       1.00      1.00      1.00      2900
           2       0.83      0.91      0.87        11
           3       0.60      1.00      0.75         3
           4       1.00      1.00      1.00      6939
           5       1.00      0.97      0.99        34
           6       1.00      1.00      1.00      1428
           7       1.00      1.00      1.00       793
           8       1.00      1.00      1.00      5736
           9       0.40      0.67      0.50         9
          10       1.00      1.00      1.00       203
          11       0.00      0.00      0.00         8
          12       0.60      0.18      0.27      7287
          13       0.00      0.00      0.00         2
          14       1.00      1.00      1.00      1812

    accuracy                           0.96    164670
   macro avg       0.76      0.78      0.76    164670
weighted avg       0.95   

In [22]:
joblib.dump(
    xgb,
    "/kaggle/working/xgboost.pkl"
)

['/kaggle/working/xgboost.pkl']

In [23]:
encoder.classes_

array(['Benign', 'Bot', 'Brute Force -Web', 'Brute Force -XSS',
       'DDOS attack-HOIC', 'DDOS attack-LOIC-UDP',
       'DDoS attacks-LOIC-HTTP', 'DoS attacks-GoldenEye',
       'DoS attacks-Hulk', 'DoS attacks-SlowHTTPTest',
       'DoS attacks-Slowloris', 'FTP-BruteForce', 'Infilteration',
       'SQL Injection', 'SSH-Bruteforce'], dtype=object)

In [24]:
normal_data = X_train_scaled[
    y_train==0
]

In [25]:
iso = IsolationForest(

    n_estimators=200,

    contamination=0.1,

    random_state=42

)


iso.fit(
    normal_data
)

IsolationForest(contamination=0.1, n_estimators=200, random_state=42)

Above is the isolation forest

This model detects unknown attacks.

IDS systems need this.

Find benign index

In [26]:
joblib.dump(
    iso,
    "/kaggle/working/isolation.pkl"
)

['/kaggle/working/isolation.pkl']

In [27]:
import numpy as np


ensemble_pred=[]


for r,x in zip(
    rf_pred,
    xgb_pred
):

    if r==x:
        ensemble_pred.append(r)

    else:

        # trust xgboost
        ensemble_pred.append(x)


ensemble_pred=np.array(
    ensemble_pred
)

In [28]:
print(
classification_report(
    y_test,
    ensemble_pred
)
)

              precision    recall  f1-score   support

           0       0.96      0.99      0.98    137505
           1       1.00      1.00      1.00      2900
           2       0.83      0.91      0.87        11
           3       0.60      1.00      0.75         3
           4       1.00      1.00      1.00      6939
           5       1.00      0.97      0.99        34
           6       1.00      1.00      1.00      1428
           7       1.00      1.00      1.00       793
           8       1.00      1.00      1.00      5736
           9       0.40      0.67      0.50         9
          10       1.00      1.00      1.00       203
          11       0.00      0.00      0.00         8
          12       0.60      0.18      0.27      7287
          13       0.00      0.00      0.00         2
          14       1.00      1.00      1.00      1812

    accuracy                           0.96    164670
   macro avg       0.76      0.78      0.76    164670
weighted avg       0.95   

In [29]:
metrics={


"Random Forest":{

"accuracy":accuracy_score(
    y_test,
    rf_pred
)

},


"XGBoost":{

"accuracy":accuracy_score(
    y_test,
    xgb_pred
)

}

}

In [30]:
import json
with open(
"/kaggle/working/metrics.json",
"w"
) as f:

    json.dump(
        metrics,
        f
    )

In [31]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import json


metrics = {

    "Random Forest": {

        "accuracy": accuracy_score(
            y_test,
            rf_pred
        ),

        "precision": precision_score(
            y_test,
            rf_pred,
            average="weighted"
        ),

        "recall": recall_score(
            y_test,
            rf_pred,
            average="weighted"
        ),

        "f1_score": f1_score(
            y_test,
            rf_pred,
            average="weighted"
        )
    },


    "XGBoost": {

        "accuracy": accuracy_score(
            y_test,
            xgb_pred
        ),

        "precision": precision_score(
            y_test,
            xgb_pred,
            average="weighted"
        ),

        "recall": recall_score(
            y_test,
            xgb_pred,
            average="weighted"
        ),

        "f1_score": f1_score(
            y_test,
            xgb_pred,
            average="weighted"
        )
    }
}

In [32]:
with open(
    "/kaggle/working/metrics.json",
    "w"
) as f:

    json.dump(
        metrics,
        f,
        indent=4
    )

In [33]:
with open(
    "/kaggle/working/metrics.json"
) as f:

    print(
        json.load(f)
    )

{'Random Forest': {'accuracy': 0.9553592032549948, 'precision': 0.9410829030428708, 'recall': 0.9553592032549948, 'f1_score': 0.9436858521973817}, 'XGBoost': {'accuracy': 0.9582073237383859, 'precision': 0.9470172532110852, 'recall': 0.9582073237383859, 'f1_score': 0.9472691838788199}}


In [34]:
!ls /kaggle/working

features.pkl   metrics.json	   random_forest.pkl  xgboost.pkl
isolation.pkl  __notebook__.ipynb  scaler.pkl


In [35]:
import os

os.listdir("/kaggle/working")

['scaler.pkl',
 '__notebook__.ipynb',
 'isolation.pkl',
 'features.pkl',
 'random_forest.pkl',
 'xgboost.pkl',
 'metrics.json']

In [36]:
import shutil


shutil.copy(
    ENCODER_PATH,
    "/kaggle/working/label_encoder.pkl"
)

'/kaggle/working/label_encoder.pkl'

In [37]:
os.listdir("/kaggle/working")

['scaler.pkl',
 '__notebook__.ipynb',
 'isolation.pkl',
 'features.pkl',
 'random_forest.pkl',
 'xgboost.pkl',
 'metrics.json',
 'label_encoder.pkl']